
# Saudi Master Data Generator — Self-contained Notebook

This notebook implements the uploaded Saudi master-data generation specification. It creates these DataFrames:

- `territory_df`
- `salesperson_df`
- `van_df`
- `customer_df`
- `holiday_df`
- `config_df`
- `rfm_scores_df`

It also validates foreign keys and business rules, then writes CSV outputs plus `data_quality_report.json`.


In [1]:

# Self-contained Saudi master/reference data generator  — v2 (cold-truck bifurcation + percentile RFM)
# Run this notebook top-to-bottom. Writes outputs into ./saudi_master_data_output

import sys, subprocess, importlib.util
from pathlib import Path
from datetime import date, datetime, timedelta
import json, math, random, string

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("pandas")
ensure_package("numpy")
ensure_package("faker")

import numpy as np
import pandas as pd
from faker import Faker

# ─────────────────────────────────────────────────────────────────
# Holiday CSV path — set this to your holiday file before running
# ─────────────────────────────────────────────────────────────────
HOLIDAY_CSV_PATH = "D:\Data Science\Basamh\JP_Yash\journey-planner\saudi_master_data_output\holiday_1.csv"   # ← update path if needed

try:
    fake = Faker(["ar_SA", "en_US"])
except Exception:
    fake = Faker("en_US")

# ─────────────────────────────────────────────────────────────────
# Static specification inputs
# ─────────────────────────────────────────────────────────────────

TERRITORIES = [
    {"territory_id": "TER_RUH", "territory_name": "Riyadh Central",
     "center_lat": 24.7136, "center_lng": 46.6753, "radius_km": 25,
     "warehouse_lat": 24.5790, "warehouse_lng": 46.8237, "warehouse_address": "Industrial Area, Riyadh"},
    {"territory_id": "TER_JED", "territory_name": "Jeddah North",
     "center_lat": 21.5433, "center_lng": 39.1728, "radius_km": 22,
     "warehouse_lat": 21.3429, "warehouse_lng": 39.2357, "warehouse_address": "Al Khomrah Logistics Area, Jeddah"},
    {"territory_id": "TER_DMM", "territory_name": "Dammam Metro",
     "center_lat": 26.4207, "center_lng": 50.0888, "radius_km": 20,
     "warehouse_lat": 26.2926, "warehouse_lng": 50.1629, "warehouse_address": "2nd Industrial City, Dammam"},
]

LOCALITIES = {
    "TER_RUH": [("Olaya",24.7115,46.6746),("Al Malaz",24.6676,46.7351),
                ("Al Sulaymaniyah",24.7012,46.7112),("Al Yasmin",24.8271,46.6302),("Hittin",24.7636,46.6022)],
    "TER_JED": [("Al Rawdah",21.5656,39.1652),("Al Hamra",21.5262,39.1611),
                ("Al Safa",21.5854,39.2181),("Al Salamah",21.5948,39.1485),("Al Zahra",21.6152,39.1335)],
    "TER_DMM": [("Al Faisaliyah",26.4282,50.0786),("Al Shati",26.4701,50.1124),
                ("Al Mazruiyah",26.4481,50.0962),("Al Badiyah",26.4021,50.0587),("Al Nuzha",26.4337,50.0433)],
}

CITY_CODES    = {"TER_RUH": "RUH", "TER_JED": "JED", "TER_DMM": "DMM"}
PLATE_PREFIXES = {"TER_RUH": "RU",  "TER_JED": "JE",  "TER_DMM": "DM"}

SAUDI_SALESPERSON_NAMES = [
    "Abdullah Al-Qahtani","Fahad Al-Otaibi","Mohammed Al-Harbi","Nasser Al-Dossari","Khalid Al-Ghamdi",
    "Saeed Al-Zahrani","Yousef Al-Mutairi","Majed Al-Shammari","Ahmed Al-Anazi","Salem Al-Rashidi",
    "Omar Al-Shehri","Hassan Al-Yami","Rashid Al-Malki","Ibrahim Al-Subaie","Mansour Al-Qahtani",
    "Waleed Al-Harbi","Bilal Khan","Imran Ahmed","Sameer Khan","Nadeem Ali","Arif Rahman","Mustafa Hussain",
]
OWNER_NAMES = [
    "Al Rajhi","Al Othman","Al Harbi","Al Qahtani","Al Ghamdi","Al Zahrani","Al Dossari",
    "Al Mutairi","Al Shammari","Al Anazi","Al Rashid","Al Saleh","Al Malki","Al Subaie","Al Shehri",
]
BUSINESS_PREFIXES = [
    "Al Noor","Al Waha","Al Baraka","Al Safa","Al Madina","Al Qassim","Al Riyadh",
    "Al Jazeera","Al Khaleej","Al Nada","Al Nakheel","Al Rawabi","Al Tazaj","Al Dana","Al Manar",
]
SHOP_CATEGORIES = [
    "Grocery","Mini Market","Supermarket","Restaurant","Cafe","Bakery",
    "Butchery","Cold Store","Hotel Kitchen","Catering Kitchen",
]
BUSINESS_SUFFIX_BY_CATEGORY = {
    "Grocery":          ["Grocery","Baqala","Food Store"],
    "Mini Market":      ["Mini Market","Corner Market","Baqala"],
    "Supermarket":      ["Supermarket","Hyper Mini","Market"],
    "Restaurant":       ["Restaurant","Kitchen","Grill"],
    "Cafe":             ["Cafe","Coffee House","Roastery"],
    "Bakery":           ["Bakery","Sweets & Bakery","Oven"],
    "Butchery":         ["Butchery","Meat Shop","Fresh Meat"],
    "Cold Store":       ["Cold Store","Frozen Foods","Chilled Foods"],
    "Hotel Kitchen":    ["Hotel Kitchen","Hospitality Supplies"],
    "Catering Kitchen": ["Catering Kitchen","Banquet Kitchen"],
}
VISIT_DAYS       = ["Saturday","Sunday","Monday","Tuesday","Wednesday","Thursday"]
ORDER_WINDOWS    = ["Morning","Midday","Afternoon"]
LIFECYCLE_STATES = ["Active","New","At Risk","Dormant","Churned"]
LIFECYCLE_PROBS  = [0.65, 0.10, 0.15, 0.08, 0.02]

# ─────────────────────────────────────────────────────────────────
# Utility functions
# ─────────────────────────────────────────────────────────────────

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); Faker.seed(seed)

def haversine_km(lat1, lng1, lat2, lng2):
    r = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi, dlambda = math.radians(lat2-lat1), math.radians(lng2-lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*r*math.atan2(math.sqrt(a), math.sqrt(1-a))

def jitter_location(lat, lng, radius_km=2.5):
    lat_j = np.random.normal(0, radius_km/111/2)
    lng_j = np.random.normal(0, radius_km/(111*np.cos(np.radians(lat)))/2)
    return float(lat+lat_j), float(lng+lng_j)

def random_date_between(start, end):
    return start + timedelta(days=random.randint(0, (end-start).days))

def weighted_choice(values, probs):
    return random.choices(values, weights=probs, k=1)[0]

def generate_plate(prefix):
    return f"{prefix}{random.choice(string.ascii_uppercase)} {random.randint(1000,9999)}"

def performance_multiplier():
    b = random.random()
    if b < 0.15:  return round(random.uniform(1.10,1.20),2)
    if b < 0.85:  return round(random.uniform(0.95,1.08),2)
    return round(random.uniform(0.85,0.94),2)

def cold_truck_required(category):
    """
    Deterministic: only Cold Store and Butchery require cold-chain delivery.
    All other categories use a normal truck.
    """
    return category in {"Cold Store", "Butchery"}

def choose_payment_type(volume_tier):
    return "credit" if random.random() < {"HIGH":0.75,"MED":0.45,"LOW":0.20}[volume_tier] else "cash"

def generate_credit(volume_tier, payment_type, lifecycle_state):
    if payment_type == "cash": return 0.0, 0.0
    lo, hi = {"HIGH":(30000,120000),"MED":(10000,45000),"LOW":(2000,15000)}[volume_tier]
    credit_limit = round(random.uniform(lo, hi), 2)
    if lifecycle_state in ["At Risk","Dormant"]: pct = random.uniform(0.55,1.10)
    elif lifecycle_state == "Churned":           pct = random.uniform(0.70,1.10)
    else:
        pct = random.choices(
            [random.uniform(0.05,0.35), random.uniform(0.35,0.70), random.uniform(0.70,1.10)],
            weights=[0.65,0.25,0.10], k=1)[0]
    return credit_limit, round(credit_limit*pct, 2)

def generate_shop_name(category, locality, used_names):
    suffix = random.choice(BUSINESS_SUFFIX_BY_CATEGORY[category])
    include_locality = random.random() < 0.28
    templates = ["{prefix} {suffix}","{owner} {suffix}","{prefix} Fresh {suffix}","{owner} Trading {suffix}"]
    if include_locality:
        templates.extend(["{locality} {suffix}","{prefix} {locality} {suffix}"])
    if suffix in {"Frozen Foods","Cold Store","Chilled Foods"} and category != "Cold Store":
        suffix = random.choice(BUSINESS_SUFFIX_BY_CATEGORY[category])
    for _ in range(50):
        name = random.choice(templates).format(
            prefix=random.choice(BUSINESS_PREFIXES), owner=random.choice(OWNER_NAMES),
            locality=locality, suffix=suffix)
        if name not in used_names:
            used_names.add(name); return name
    name = f"{locality} {suffix} {random.randint(100,999)}"
    used_names.add(name); return name


# ─────────────────────────────────────────────────────────────────
# Generator functions
# ─────────────────────────────────────────────────────────────────

def generate_territories():
    df = pd.DataFrame(TERRITORIES).copy()
    df["default_salesperson"] = None; df["default_van"] = None
    return df[["territory_id","territory_name","warehouse_lat","warehouse_lng","warehouse_address",
               "center_lat","center_lng","radius_km","default_salesperson","default_van"]]

def generate_salespeople(territory_df, salespeople_per_territory=3):
    names = SAUDI_SALESPERSON_NAMES.copy(); random.shuffle(names); rows = []
    for _, ter in territory_df.iterrows():
        code = CITY_CODES[ter.territory_id]
        for i in range(1, salespeople_per_territory+1):
            rows.append({"sales_id": f"SAL_{code}_{i:03d}", "name": names.pop() if names else fake.name(),
                         "territory_id": ter.territory_id, "working_hours_per_day": 8.0,
                         "shift_start_time": "09:00", "assigned_van": None,
                         "active_status": random.random()<0.93, "performance_multiplier": performance_multiplier()})
    return pd.DataFrame(rows)

def generate_vans(territory_df, salesperson_df, backup_vans_per_territory=1):
    """
    Van generation with guaranteed cold-truck availability per territory.
    At least 40% of vans in each territory are cold-truck enabled so that
    cold-chain customers always have a valid van to be assigned to.
    """
    rows = []
    for _, ter in territory_df.iterrows():
        n_sales   = int((salesperson_df["territory_id"] == ter.territory_id).sum())
        total     = n_sales + backup_vans_per_territory
        code      = CITY_CODES[ter.territory_id]
        prefix    = PLATE_PREFIXES[ter.territory_id]
        # Guarantee at least ceil(40% of vans) are cold-enabled
        cold_guaranteed = math.ceil(total * 0.40)
        cold_flags = [True]*cold_guaranteed + [False]*(total - cold_guaranteed)
        random.shuffle(cold_flags)
        for i in range(1, total+1):
            rows.append({"van_id": f"VAN_{code}_{i:03d}", "registration_no": generate_plate(prefix),
                         "cold_truck_enabled": cold_flags[i-1],
                         "territory_id": ter.territory_id, "active_status": random.random()<0.96})
    return pd.DataFrame(rows)

def assign_vans_to_salespeople(salesperson_df, van_df):
    salesperson_df = salesperson_df.copy()
    for territory_id in salesperson_df["territory_id"].unique():
        sp_idx = salesperson_df.index[salesperson_df["territory_id"]==territory_id].tolist()
        vans   = van_df[van_df["territory_id"]==territory_id]["van_id"].tolist()
        for idx, van_id in zip(sp_idx, vans):
            salesperson_df.loc[idx, "assigned_van"] = van_id
    return salesperson_df

def assign_defaults(territory_df, salesperson_df, van_df):
    territory_df = territory_df.copy()
    for idx, ter in territory_df.iterrows():
        tid = ter.territory_id
        active_sp = salesperson_df[(salesperson_df.territory_id==tid)&(salesperson_df.active_status)]
        sp_pool   = active_sp if not active_sp.empty else salesperson_df[salesperson_df.territory_id==tid]
        default_sp  = sp_pool.sort_values("performance_multiplier",ascending=False).iloc[0]
        default_van = default_sp.assigned_van
        if pd.isna(default_van):
            default_van = van_df[van_df.territory_id==tid].iloc[0].van_id
        territory_df.loc[idx,"default_salesperson"] = default_sp.sales_id
        territory_df.loc[idx,"default_van"]         = default_van
    return territory_df

def generate_customers(territory_df, customers_per_territory=100):
    rows = []; today = date.today()
    start_acq = today - timedelta(days=5*365)
    end_acq   = today - timedelta(days=30)
    for _, ter in territory_df.iterrows():
        tid  = ter.territory_id; code = CITY_CODES[tid]; used_names = set()
        tiers = ["HIGH"]*20 + ["MED"]*30 + ["LOW"]*50; random.shuffle(tiers)
        for i in range(1, customers_per_territory+1):
            locality, base_lat, base_lng = random.choice(LOCALITIES[tid])
            for _ in range(100):
                lat, lng = jitter_location(base_lat, base_lng, radius_km=2.5)
                if haversine_km(lat,lng,ter.center_lat,ter.center_lng) <= ter.radius_km: break
            else:
                lat, lng = base_lat, base_lng
            tier      = tiers[i-1]
            lifecycle = weighted_choice(LIFECYCLE_STATES, LIFECYCLE_PROBS)
            category  = random.choice(SHOP_CATEGORIES)
            payment_type = choose_payment_type(tier)
            credit_limit, outstanding_balance = generate_credit(tier, payment_type, lifecycle)
            customer_rating = int(random.choices([1,2,3,4,5],weights=[0.05,0.10,0.25,0.35,0.25],k=1)[0])
            rows.append({
                "customer_id":          f"CUS_{code}_{i:04d}",
                "shop_name":            generate_shop_name(category, locality, used_names),
                "gps_lat":              round(lat,6),
                "gps_lng":              round(lng,6),
                "locality":             locality,
                "territory_id":         tid,
                "customer_rating":      customer_rating,
                "review_rating":        round(float(np.clip(np.random.normal(4.0,0.55),2.5,5.0)),1),
                "shop_category":        category,
                "cold_truck_required":  cold_truck_required(category),
                "volume_tier":          tier,
                "payment_type":         payment_type,
                "credit_limit":         credit_limit,
                "outstanding_balance":  outstanding_balance,
                "lifecycle_state":      lifecycle,
                "acquisition_date":     random_date_between(start_acq, end_acq),
                "preferred_visit_day":  random.choice(VISIT_DAYS),
                "preferred_order_window": random.choice(ORDER_WINDOWS),
            })
    return pd.DataFrame(rows)

def fridays_between(start_date, end_date):
    cur = start_date
    while cur.weekday() != 4: cur += timedelta(days=1)
    while cur <= end_date: yield cur; cur += timedelta(days=7)

def load_holidays_from_csv(holiday_csv_path: str = r'D:\Data Science\Basamh\JP_Yash\journey-planner\saudi_master_data_output\holiday.csv') -> pd.DataFrame:
    """
    Load the holiday calendar from an external CSV file instead of generating it.

    Expected CSV columns (same schema as the generated holiday_df):
        holiday_id, salesperson_holiday, van_holiday,
        territory_holiday, from_date, to_date, reason

    Date columns (from_date, to_date) are parsed to datetime.date objects
    so they are compatible with the rest of the pipeline.
    """
    df = pd.read_csv(holiday_csv_path)

    # Normalise column names (strip whitespace, lowercase)
    df.columns = [c.strip() for c in df.columns]

    # Parse date columns to date objects
    for col in ["from_date", "to_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.date

    # Replace empty-string / whitespace-only cells with None
    for col in ["salesperson_holiday", "van_holiday", "territory_holiday"]:
        if col in df.columns:
            df[col] = df[col].where(df[col].notna() & (df[col].astype(str).str.strip() != ""), other=None)

    print(f"[load_holidays_from_csv] Loaded {len(df)} holiday rows from '{holiday_csv_path}'")
    return df

def generate_config():
    values = {"avg_speed_kmh":32,"avg_service_time_min":22,"buffer_pct":0.15,"rfm_window_days":90,
              "route_partial_prob":0.08,"route_cancel_prob":0.03,"traffic_jam_prob":0.12,
              "credit_outstanding_cap":0.85,"normal_shift_start_time":"09:00","ramadan_shift_start_time":"10:00"}
    return pd.DataFrame([{"config_key":k,"config_value":v} for k,v in values.items()])


# ─────────────────────────────────────────────────────────────────────────────
# RFM + Context scoring — with cold-truck group bifurcation
# ─────────────────────────────────────────────────────────────────────────────
#
# Segmentation architecture
# ─────────────────────────────────────────────────────────────────────────────
# Level 1 — Territory          (TER_RUH, TER_JED, TER_DMM)
# Level 2 — Truck Group        (cold_truck = True / False)
# Level 3 — RFM Segment        (High / Medium / Low)
#
# Percentile thresholds (within each territory × truck-group peer set):
#   Top 30%    → High
#   Next 40%   → Medium   (30th–70th percentile)
#   Bottom 30% → Low
#
# Component weights on final_customer_score:
#   rfm_combined   40%  = mean(r_score, f_score, m_score)
#   seasonality    30%
#   territory      10%
#   locality       10%
#   rating         10%
# ─────────────────────────────────────────────────────────────────────────────

def _filter_rfm_window(visit_df: pd.DataFrame, window_days: int = 90):
    analysis_date = pd.Timestamp.today().normalize()
    start_date    = analysis_date - pd.Timedelta(days=window_days-1)
    mask = pd.to_datetime(visit_df["visit_date"]).between(start_date, analysis_date)
    return visit_df[mask].copy(), analysis_date

def _min_max_scale(series: pd.Series, reverse: bool = False) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi == lo: return pd.Series(1.0, index=series.index)
    if reverse:  return (hi - series) / (hi - lo)
    return (series - lo) / (hi - lo)

def _scale_within_territory(df: pd.DataFrame, col: str, reverse: bool = False) -> pd.Series:
    """Min-max scale col within each territory_id group."""
    result = pd.Series(index=df.index, dtype=float)
    for _, grp in df.groupby("territory_id"):
        result.loc[grp.index] = _min_max_scale(grp[col], reverse=reverse).values
    return result

def _compute_seasonality_score(customer_df: pd.DataFrame, visit_df: pd.DataFrame) -> pd.Series:
    """
    Seasonality score (0-1) based on same-calendar-month buying pattern,
    scaled within each territory peer set.
    Customers with no same-month history receive the neutral score 0.5.
    """
    v = visit_df.copy(); v["visit_date"] = pd.to_datetime(v["visit_date"])
    analysis_month = v["visit_date"].max().month
    same_month = v[v["visit_date"].dt.month == analysis_month]
    sm_agg = (same_month.groupby("customer_id")
              .agg(sm_visits=("visit_date","count"), sm_success=("successful_visit","sum"))
              .reset_index())
    sm_agg["sm_success_rate"] = sm_agg["sm_success"] / sm_agg["sm_visits"].clip(lower=1)
    total_agg = v.groupby("customer_id").agg(total_visits=("visit_date","count")).reset_index()
    base   = customer_df[["customer_id","territory_id"]].copy()
    merged = base.merge(sm_agg, on="customer_id", how="left").merge(total_agg, on="customer_id", how="left")
    merged["sm_visits"]       = merged["sm_visits"].fillna(0)
    merged["sm_success_rate"] = merged["sm_success_rate"].fillna(0.0)
    merged["total_visits"]    = merged["total_visits"].fillna(1)
    merged["sm_visit_share"]  = merged["sm_visits"] / merged["total_visits"].clip(lower=1)
    has_history = (merged["sm_visits"] > 0).values
    merged["seasonality_raw"] = 0.6*merged["sm_success_rate"] + 0.4*merged["sm_visit_share"]
    scaled = _scale_within_territory(merged.reset_index(drop=True), "seasonality_raw", reverse=False)
    result = scaled.where(pd.Series(has_history), 0.5)
    result.index = merged["customer_id"].values
    return result


def _assign_segment_percentile(group_scores: pd.Series) -> pd.Series:
    """
    Assign High / Medium / Low based on percentile rank within the group.

    Thresholds:
        Top 30%          → High    (pct_rank >= 0.70)
        Middle 40%       → Medium  (0.30 <= pct_rank < 0.70)
        Bottom 30%       → Low     (pct_rank < 0.30)

    pct_rank uses method='average' so ties share equal rank; ascending=False
    means the highest score gets rank 1.0 (best → top of the group).
    """
    pct = group_scores.rank(pct=True, method="average", ascending=True)
    segments = pd.Series("Low", index=group_scores.index)
    segments[pct >= 0.70] = "High"
    segments[(pct >= 0.30) & (pct < 0.70)] = "Medium"
    return segments


def generate_rfm_scores(customer_df: pd.DataFrame,
                         visit_df: pd.DataFrame = None,
                         config_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Compute the RFM score table with cold-truck group bifurcation.

    Segmentation hierarchy
    ──────────────────────
    Territory → Truck Group (cold/non-cold) → RFM Segment (High/Medium/Low)

    Each customer is ranked and segmented only against peers who share the
    same territory AND the same cold_truck_required flag.  This means a Cold
    Store's 'High' is relative to other cold-chain customers in the same
    territory — not mixed with ambient customers.

    Percentile rules (within territory × truck group):
        Top 30%    → High
        Next 40%   → Medium
        Bottom 30% → Low
    """
    if visit_df is None:
        raise ValueError("visit_df is required to compute RFM scores.")

    window_days = 90
    if config_df is not None and not config_df.empty:
        cfg = config_df.set_index("config_key")["config_value"].to_dict()
        window_days = int(float(cfg.get("rfm_window_days", 90)))

    visit_df = visit_df.copy()
    visit_df["visit_date"] = pd.to_datetime(visit_df["visit_date"])

    # ── 1. RFM window ────────────────────────────────────────────────────────
    v_win, analysis_date = _filter_rfm_window(visit_df, window_days=window_days)
    v_succ = v_win[v_win["successful_visit"] == True].copy()

    # ── 2. Per-customer R/F/M aggregates ────────────────────────────────────
    rfm_agg = (v_succ.groupby("customer_id")
               .agg(last_successful_visit=("visit_date","max"),
                    frequency=("successful_visit","sum"),
                    monetary=("transaction_amount","sum"))
               .reset_index())
    rfm_agg["recency"] = (analysis_date - rfm_agg["last_successful_visit"]).dt.days

    # ── 3. Merge; bring in cold_truck_required from customer_df ─────────────
    base = customer_df[["customer_id","locality","territory_id",
                         "customer_rating","cold_truck_required"]].copy()
    rfm  = base.merge(rfm_agg, on="customer_id", how="left")
    rfm["recency"]               = rfm["recency"].fillna(window_days)
    rfm["frequency"]             = rfm["frequency"].fillna(0)
    rfm["monetary"]              = rfm["monetary"].fillna(0.0)
    rfm["last_successful_visit"] = rfm["last_successful_visit"].fillna(pd.NaT)

    # ── 4. cold_truck_group label (used for display & grouping) ─────────────
    rfm["cold_truck_group"] = rfm["cold_truck_required"].map(
        {True: "Cold Truck", False: "Normal Truck"})

    # ── 5. R / F / M — scaled within territory ──────────────────────────────
    rfm["r_score"] = _scale_within_territory(rfm, "recency",   reverse=True)
    rfm["f_score"] = _scale_within_territory(rfm, "frequency", reverse=False)
    rfm["m_score"] = _scale_within_territory(rfm, "monetary",  reverse=False)

    # ── 6. territory_score (global across territories) ───────────────────────
    succ_all = visit_df[visit_df["successful_visit"] == True].copy()
    succ_ter = succ_all.merge(customer_df[["customer_id","territory_id"]], on="customer_id", how="left")
    ter_avg  = (succ_ter.groupby("territory_id")["transaction_amount"]
                .mean().reset_index().rename(columns={"transaction_amount":"ter_avg_amount"}))
    ter_avg["territory_score"] = _min_max_scale(ter_avg["ter_avg_amount"])
    rfm = rfm.merge(ter_avg[["territory_id","territory_score"]], on="territory_id", how="left")
    rfm["territory_score"] = rfm["territory_score"].fillna(0.0)

    # ── 7. locality_score (within territory) ────────────────────────────────
    succ_loc = succ_all.merge(
        customer_df[["customer_id","locality","territory_id"]], on="customer_id", how="left")
    loc_avg  = (succ_loc.groupby(["territory_id","locality"])["transaction_amount"]
                .mean().reset_index().rename(columns={"transaction_amount":"loc_avg_amount"}))
    rfm = rfm.merge(loc_avg, on=["territory_id","locality"], how="left")
    rfm["loc_avg_amount"] = rfm["loc_avg_amount"].fillna(
        rfm["territory_id"].map(ter_avg.set_index("territory_id")["ter_avg_amount"]))
    rfm["locality_score"] = _scale_within_territory(rfm, "loc_avg_amount", reverse=False)

    # ── 8. rating_score (within territory) ──────────────────────────────────
    rfm["rating_score"] = _scale_within_territory(rfm, "customer_rating", reverse=False)

    # ── 9. seasonality_score (within territory) ──────────────────────────────
    seasonality_map       = _compute_seasonality_score(customer_df, visit_df)
    rfm["seasonality_score"] = rfm["customer_id"].map(seasonality_map).fillna(0.5)

    # ── 10. Final customer score ─────────────────────────────────────────────
    W_RFM, W_SEA, W_TER, W_LOC, W_RAT = 0.40, 0.30, 0.10, 0.10, 0.10
    rfm["rfm_combined"] = rfm[["r_score","f_score","m_score"]].mean(axis=1)
    rfm["final_customer_score"] = (
        W_RFM * rfm["rfm_combined"]
        + W_SEA * rfm["seasonality_score"]
        + W_TER * rfm["territory_score"]
        + W_LOC * rfm["locality_score"]
        + W_RAT * rfm["rating_score"]
    )

    # ── 11. Percentile-based RFM segment within Territory × Truck Group ──────
    #
    # Grouping key: (territory_id, cold_truck_group)
    # Within each group, sort by final_customer_score and apply:
    #     Top 30%    → High
    #     Middle 40% → Medium
    #     Bottom 30% → Low
    #
    rfm["rfm_segment_final"] = (
        rfm.groupby(["territory_id","cold_truck_group"])["final_customer_score"]
        .transform(_assign_segment_percentile)
    )

    # ── 12. Rank within Territory × Truck Group ──────────────────────────────
    rfm["customer_rank"] = (
        rfm.groupby(["territory_id","cold_truck_group"])["final_customer_score"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    rfm = rfm.sort_values(
        ["territory_id","cold_truck_group","customer_rank"]
    ).reset_index(drop=True)

    return rfm[[
        "customer_id","territory_id","cold_truck_group",
        "last_successful_visit","recency","frequency","monetary",
        "r_score","f_score","m_score","rfm_combined",
        "seasonality_score","territory_score","locality_score","rating_score",
        "final_customer_score","rfm_segment_final","customer_rank",
    ]]


# ─────────────────────────────────────────────────────────────────
# Validation and reporting
# ─────────────────────────────────────────────────────────────────

def assert_fk(child_df, child_col, parent_df, parent_col):
    missing = set(child_df[child_col].dropna()) - set(parent_df[parent_col])
    assert not missing, f"Broken FK {child_col}: {missing}"

def validate_all(territory_df, salesperson_df, van_df, customer_df,
                  holiday_df, config_df, rfm_scores_df):
    assert_fk(customer_df,  "territory_id",        territory_df,   "territory_id")
    assert_fk(salesperson_df,"territory_id",        territory_df,   "territory_id")
    assert_fk(van_df,        "territory_id",        territory_df,   "territory_id")
    assert_fk(holiday_df,    "territory_holiday",   territory_df,   "territory_id")
    assert_fk(holiday_df,    "salesperson_holiday", salesperson_df, "sales_id")
    assert_fk(holiday_df,    "van_holiday",         van_df,         "van_id")
    assert_fk(rfm_scores_df, "customer_id",         customer_df,    "customer_id")
    assert territory_df["territory_id"].is_unique
    assert salesperson_df["sales_id"].is_unique
    assert van_df["van_id"].is_unique
    assert customer_df["customer_id"].is_unique
    assert holiday_df["holiday_id"].is_unique
    assert rfm_scores_df["customer_id"].is_unique
    assert len(territory_df) == 3
    assert len(customer_df)  == 300
    assert 6  <= len(salesperson_df) <= 9
    assert 9  <= len(van_df)         <= 12
    assert len(rfm_scores_df)        == 300
    van_territory = van_df.set_index("van_id")["territory_id"].to_dict()
    sp_territory  = salesperson_df.set_index("sales_id")["territory_id"].to_dict()
    for _, sp in salesperson_df.iterrows():
        assert van_territory[sp.assigned_van] == sp.territory_id
    for _, ter in territory_df.iterrows():
        assert sp_territory[ter.default_salesperson] == ter.territory_id
        assert van_territory[ter.default_van]        == ter.territory_id
    cash   = customer_df["payment_type"] == "cash"
    credit = customer_df["payment_type"] == "credit"
    assert (customer_df.loc[cash,   "credit_limit"]        == 0).all()
    assert (customer_df.loc[cash,   "outstanding_balance"] == 0).all()
    assert (customer_df.loc[credit, "credit_limit"]         > 0).all()
    assert customer_df.loc[customer_df.shop_category.isin(["Cold Store","Butchery"]),
                           "cold_truck_required"].all()
    ter_lookup = territory_df.set_index("territory_id")
    for _, c in customer_df.iterrows():
        ter  = ter_lookup.loc[c.territory_id]
        dist = haversine_km(c.gps_lat,c.gps_lng,ter.center_lat,ter.center_lng)
        assert dist <= ter.radius_km+1e-6, f"Customer outside radius: {c.customer_id} {dist:.2f}km"
    dupes = customer_df.duplicated(subset=["territory_id","shop_name"]).sum()
    assert dupes == 0, f"Duplicate shop names inside territory: {dupes}"
    tier_counts = customer_df.groupby(["territory_id","volume_tier"]).size().unstack(fill_value=0)
    for tid in territory_df["territory_id"]:
        assert int(tier_counts.loc[tid,"HIGH"]) == 20
        assert int(tier_counts.loc[tid,"MED"])  == 30
        assert int(tier_counts.loc[tid,"LOW"])  == 50
    return True

def data_quality_report(territory_df, salesperson_df, van_df, customer_df,
                         holiday_df, config_df, rfm_scores_df):
    # Segment counts broken down by territory × truck group
    seg_breakdown = (
        rfm_scores_df.groupby(["territory_id","cold_truck_group","rfm_segment_final"])
        .size().reset_index(name="count")
        .to_dict(orient="records")
    )
    report = {
        "row_counts": {
            "territory": len(territory_df), "salesperson": len(salesperson_df),
            "van": len(van_df), "customer": len(customer_df),
            "holiday": len(holiday_df), "config": len(config_df),
            "rfm_scores": len(rfm_scores_df),
        },
        "customer_tiers_by_territory": (
            customer_df.groupby(["territory_id","volume_tier"])
            .size().unstack(fill_value=0).to_dict(orient="index")),
        "cold_truck_share_by_territory": (
            customer_df.groupby("territory_id")["cold_truck_required"]
            .mean().round(3).to_dict()),
        "rfm_segments_overall": rfm_scores_df["rfm_segment_final"].value_counts().to_dict(),
        "rfm_segments_by_territory_and_truck_group": seg_breakdown,
        "lifecycle_distribution": (
            customer_df["lifecycle_state"].value_counts(normalize=True).round(3).to_dict()),
        "payment_type_distribution": (
            customer_df["payment_type"].value_counts(normalize=True).round(3).to_dict()),
        "validation_status": "passed",
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }
    return report


# ─────────────────────────────────────────────────────────────────────────────
# Visit Table Generator  (cold-truck-aware van assignment + RFM quartile)
# ─────────────────────────────────────────────────────────────────────────────

def _expand_holiday_dates(holiday_df: pd.DataFrame) -> dict:
    territory_blocked = {}; salesperson_blocked = {}; van_blocked_d = {}
    for _, row in holiday_df.iterrows():
        fd = pd.to_datetime(row["from_date"]).date()
        td = pd.to_datetime(row["to_date"]).date()
        dates = set(); cur = fd
        while cur <= td: dates.add(cur); cur += timedelta(days=1)
        if pd.notna(row["territory_holiday"]) and row["territory_holiday"]:
            territory_blocked.setdefault(row["territory_holiday"], set()).update(dates)
        if pd.notna(row["salesperson_holiday"]) and row["salesperson_holiday"]:
            salesperson_blocked.setdefault(row["salesperson_holiday"], set()).update(dates)
        if pd.notna(row["van_holiday"]) and row["van_holiday"]:
            van_blocked_d.setdefault(row["van_holiday"], set()).update(dates)
    return {"territory_blocked": territory_blocked,
            "salesperson_blocked": salesperson_blocked,
            "van_blocked": van_blocked_d}


def generate_visits(customer_df, salesperson_df, van_df, territory_df,
                    holiday_df=None, rfm_scores_df=None,
                    start_date=None, end_date=None, seed=42):
    """
    Generate synthetic visit records with two key enhancements:

    1. Cold-truck-aware van assignment
    ───────────────────────────────────
    • Cold-truck customers (cold_truck_required=True) are only served by
      cold-enabled vans (cold_truck_enabled=True).
    • Normal customers may be served by any active van in the territory.
    • If no cold-enabled van is available in a territory the customer is
      skipped for that date (graceful degradation, not a crash).

    2. Visit frequency driven by RFM quartile
    ──────────────────────────────────────────
    Customers are visited every week by default, but their effective visit
    frequency is modulated by their RFM quartile within their
    Territory × Truck Group peer set:

        Quartile 4 (top 25% of final_customer_score)  → visit_freq_multiplier = 1.0
        Quartile 3                                     → 0.85   (skip ~15% of weeks)
        Quartile 2                                     → 0.65   (skip ~35% of weeks)
        Quartile 1 (bottom 25%)                        → 0.45   (skip ~55% of weeks)

    The multiplier is applied as a Bernoulli trial per candidate visit date:
        random.random() < visit_freq_multiplier  →  attempt the visit

    This means High-RFM customers receive more touchpoints per quarter
    than Low-RFM customers, reflecting real-world prioritisation by the
    sales force.

    Visit columns
    ─────────────
    visit_id            str   – unique per territory, e.g. VIS_RUH_000001
    visit_date          date  – Sat-Thu only; Fridays and holidays are skipped
    customer_id         str   – FK → customer_df
    sales_id            str   – FK → salesperson_df
    van_id              str   – FK → van_df (cold-enabled iff customer needs cold)
    cold_truck_required bool  – denormalised from customer_df for quick filtering
    rfm_segment         str   – denormalised from rfm_scores_df (High/Medium/Low/"Unscored")
    rfm_quartile        int   – 1..4 within Territory × Truck Group (4 = best)
    successful_visit    bool  – whether a transaction occurred
    transaction_amount  float – SAR; 0.0 when unsuccessful
    notes               str   – free-text remark
    """
    random.seed(seed); np.random.seed(seed)
    today = date.today()
    if start_date is None: start_date = today - timedelta(days=480)
    if end_date   is None: end_date   = today - timedelta(days=1)

    # ── Holiday sets ─────────────────────────────────────────────────────────
    if holiday_df is not None and len(holiday_df) > 0:
        blocked = _expand_holiday_dates(holiday_df)
    else:
        blocked = {"territory_blocked":{}, "salesperson_blocked":{}, "van_blocked":{}}
    ter_blocked = blocked["territory_blocked"]
    sp_blocked  = blocked["salesperson_blocked"]
    van_blk     = blocked["van_blocked"]

    # ── Build staff pools: territory → {cold: [(sp, van)], normal: [(sp, van)]} ─
    van_cold_map = van_df.set_index("van_id")["cold_truck_enabled"].to_dict()
    van_active   = set(van_df[van_df["active_status"]]["van_id"])

    staff_cold   = {}   # territory_id → list of (sales_id, van_id)  — cold-enabled vans
    staff_normal = {}   # territory_id → list of (sales_id, van_id)  — any active van

    for ter_id, grp in (salesperson_df[salesperson_df["active_status"]]
                         .merge(van_df[van_df["active_status"]][["van_id","territory_id","cold_truck_enabled"]],
                                left_on=["territory_id","assigned_van"],
                                right_on=["territory_id","van_id"], how="left")
                         .groupby("territory_id")):
        pairs = list(zip(grp["sales_id"], grp["van_id"]))
        staff_normal[ter_id] = pairs
        staff_cold[ter_id]   = [(s, v) for s, v in pairs if van_cold_map.get(v, False)]

    # ── RFM quartile map: customer_id → (segment, quartile) ──────────────────
    cust_rfm = {}   # customer_id → {"segment": str, "quartile": int}
    if rfm_scores_df is not None and not rfm_scores_df.empty:
        tmp = rfm_scores_df[["customer_id","territory_id","cold_truck_group",
                              "rfm_segment_final","final_customer_score"]].copy()
        # Compute quartile within Territory × Truck Group  (4 = best)
        tmp["rfm_quartile"] = (
            tmp.groupby(["territory_id","cold_truck_group"])["final_customer_score"]
            .transform(lambda s: pd.qcut(s.rank(method="first"), 4,
                                          labels=[1,2,3,4]).astype(int))
        )
        for _, row in tmp.iterrows():
            cust_rfm[row["customer_id"]] = {
                "segment":  row["rfm_segment_final"],
                "quartile": int(row["rfm_quartile"]),
            }

    # Visit frequency multiplier by RFM quartile
    quartile_freq = {4: 1.00, 3: 0.85, 2: 0.65, 1: 0.45}

    success_probs = {"HIGH": 0.92, "MED": 0.82, "LOW": 0.70}
    amount_params = {"HIGH": (12000,4000), "MED": (3500,1200), "LOW": (700,300)}

    successful_notes = [
        "Order placed successfully.","Full order confirmed.","Client satisfied with delivery.",
        "Repeat order; no issues.","New SKUs added to order.","Paid on delivery.",
        "Invoice issued.","Seasonal top-up order.","Promotional items ordered.",
        "Standard visit; order processed.",
    ]
    unsuccessful_notes = [
        "Shop closed at time of visit.","Customer not available.",
        "Stock sufficient; no order needed.","Payment overdue; order withheld.",
        "Customer requested reschedule.","Cold-chain issue; order deferred.",
        "Order cancelled by customer.","Credit limit exceeded.",
        "Competitor promotion; no order.","No decision-maker present.",
    ]
    day_map = {"Saturday":5,"Sunday":6,"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3}

    rows = []; visit_counter = {}

    for _, cust in customer_df.iterrows():
        ter_id      = cust["territory_id"]
        needs_cold  = bool(cust["cold_truck_required"])

        # Choose appropriate staff pool
        if needs_cold:
            pool = staff_cold.get(ter_id, [])
        else:
            pool = staff_normal.get(ter_id, [])
        if not pool:
            continue  # no suitable van available in territory

        target_wd = day_map.get(cust["preferred_visit_day"])
        if target_wd is None: continue

        # Candidate dates aligned to customer's preferred weekday
        cur = start_date
        while cur.weekday() != target_wd: cur += timedelta(days=1)
        visit_dates = []
        while cur <= end_date: visit_dates.append(cur); cur += timedelta(days=7)

        sp_id, van_id = random.choice(pool)
        tier      = cust["volume_tier"]
        lifecycle = cust.get("lifecycle_state","Active")

        lifecycle_adj = {"Active":0.00,"New":+0.05,"At Risk":-0.15,"Dormant":-0.30,"Churned":-0.50}
        adj_prob = float(np.clip(success_probs[tier] + lifecycle_adj.get(lifecycle,0), 0.05, 0.99))
        mu, sigma = amount_params[tier]
        city_code = ter_id.replace("TER_","")
        visit_counter.setdefault(ter_id, 0)

        # RFM metadata for this customer
        rfm_info  = cust_rfm.get(cust["customer_id"], {"segment":"Unscored","quartile":2})
        rfm_seg   = rfm_info["segment"]
        rfm_q     = rfm_info["quartile"]
        freq_mult = quartile_freq.get(rfm_q, 0.65)

        for vdate in visit_dates:
            # ── Holiday checks ────────────────────────────────────────────────
            if vdate in ter_blocked.get(ter_id, set()):  continue
            if vdate in sp_blocked.get(sp_id, set()):    continue
            if vdate in van_blk.get(van_id, set()):      continue

            # ── RFM-quartile frequency gate ───────────────────────────────────
            # Higher quartile → more visits attempted.  This ensures that the
            # final visit data reflects priority given to high-value customers.
            if random.random() > freq_mult:
                continue   # skip this week's visit for lower-quartile customers

            visit_counter[ter_id] += 1
            successful = random.random() < adj_prob
            if successful:
                amount = round(float(np.clip(np.random.normal(mu,sigma), mu*0.1, mu*3.5)), 2)
                note   = random.choice(successful_notes)
            else:
                amount = 0.0
                note   = random.choice(unsuccessful_notes)

            rows.append({
                "visit_id":           f"VIS_{city_code}_{visit_counter[ter_id]:06d}",
                "visit_date":         vdate,
                "customer_id":        cust["customer_id"],
                "sales_id":           sp_id,
                "van_id":             van_id,
                "cold_truck_required": needs_cold,
                "rfm_segment":        rfm_seg,
                "rfm_quartile":       rfm_q,
                "successful_visit":   successful,
                "transaction_amount": amount,
                "notes":              note,
            })

    visit_df = pd.DataFrame(rows)
    visit_df = visit_df.sort_values(["visit_date","visit_id"]).reset_index(drop=True)
    return visit_df


# ─────────────────────────────────────────────────────────────────
# Orchestrator
# ─────────────────────────────────────────────────────────────────

def generate_all(seed=42, output_dir="jp_data_output", write_files=True):
    set_seed(seed)

    territory_df   = generate_territories()
    salesperson_df = generate_salespeople(territory_df)
    van_df         = generate_vans(territory_df, salesperson_df)
    salesperson_df = assign_vans_to_salespeople(salesperson_df, van_df)
    territory_df   = assign_defaults(territory_df, salesperson_df, van_df)
    customer_df    = generate_customers(territory_df)
    holiday_df     = load_holidays_from_csv(HOLIDAY_CSV_PATH)
    config_df      = generate_config()

    # Pass 1 — visits without RFM (uniform frequency) to seed RFM computation
    visit_df_seed = generate_visits(
        customer_df, salesperson_df, van_df, territory_df,
        holiday_df=holiday_df, rfm_scores_df=None, seed=seed)

    # Compute RFM scores from seed visits
    rfm_scores_df = generate_rfm_scores(customer_df, visit_df=visit_df_seed, config_df=config_df)

    # Pass 2 — visits with RFM quartile-driven frequency
    visit_df = generate_visits(
        customer_df, salesperson_df, van_df, territory_df,
        holiday_df=holiday_df, rfm_scores_df=rfm_scores_df, seed=seed+1)

    validate_all(territory_df, salesperson_df, van_df, customer_df,
                 holiday_df, config_df, rfm_scores_df)
    report = data_quality_report(territory_df, salesperson_df, van_df, customer_df,
                                  holiday_df, config_df, rfm_scores_df)

    outputs = {
        "territory": territory_df, "salesperson": salesperson_df, "van": van_df,
        "customer": customer_df, "holiday": holiday_df, "config": config_df,
        "rfm_scores": rfm_scores_df, "visit": visit_df,
    }

    if write_files:
        out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
        excel_path = out / "jp_data.xlsx"
        with pd.ExcelWriter (excel_path, engine = "openpyxl") as writer:
            for name, df in outputs.items():
                df.to_excel(excel_writer = writer, sheet_name=name, index = False)
                
        with open(out/"data_quality_report.json","w",encoding="utf-8") as f:
            json.dump(report, f, indent=2, default=str, ensure_ascii=False)
        print(f"Wrote CSV/JSON outputs to: {out.resolve()}")

    return outputs, report

# ── Run ──────────────────────────────────────────────────────────
outputs, report = generate_all(seed=42)
territory_df   = outputs["territory"]
salesperson_df = outputs["salesperson"]
van_df         = outputs["van"]
customer_df    = outputs["customer"]
holiday_df     = outputs["holiday"]
config_df      = outputs["config"]
rfm_scores_df  = outputs["rfm_scores"]
visit_df       = outputs["visit"]

print(json.dumps(report, indent=2, default=str, ensure_ascii=False))


<>:25: SyntaxWarning: invalid escape sequence '\D'
<>:25: SyntaxWarning: invalid escape sequence '\D'
C:\Users\rishabh.nahar\AppData\Local\Temp\ipykernel_13088\2742438006.py:25: SyntaxWarning: invalid escape sequence '\D'
  HOLIDAY_CSV_PATH = "D:\Data Science\Basamh\JP_Yash\journey-planner\saudi_master_data_output\holiday_1.csv"   # ← update path if needed


[load_holidays_from_csv] Loaded 98 holiday rows from 'D:\Data Science\Basamh\JP_Yash\journey-planner\saudi_master_data_output\holiday_1.csv'
Wrote CSV/JSON outputs to: D:\Data Science\Basamh\JP_Yash\journey-planner\jp_data_output
{
  "row_counts": {
    "territory": 3,
    "salesperson": 9,
    "van": 12,
    "customer": 300,
    "holiday": 98,
    "config": 10,
    "rfm_scores": 300
  },
  "customer_tiers_by_territory": {
    "TER_DMM": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    },
    "TER_JED": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    },
    "TER_RUH": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    }
  },
  "cold_truck_share_by_territory": {
    "TER_DMM": 0.23,
    "TER_JED": 0.22,
    "TER_RUH": 0.16
  },
  "rfm_segments_overall": {
    "Medium": 120,
    "High": 93,
    "Low": 87
  },
  "rfm_segments_by_territory_and_truck_group": [
    {
      "territory_id": "TER_DMM",
      "cold_truck_group": "Cold Truck",
      "rfm_segment_final": "Hi

In [2]:

# ── Visit table overview ─────────────────────────────────────────────────────
print(f"visit_df shape  : {visit_df.shape}")
print(f"Date range      : {visit_df['visit_date'].min()}  →  {visit_df['visit_date'].max()}")
print(f"Success rate    : {visit_df['successful_visit'].mean():.1%}")
print(f"Total revenue   : SAR {visit_df['transaction_amount'].sum():,.0f}")
print()
print("Cold-truck vs Normal-truck visit split:")
display(
    visit_df.groupby("cold_truck_required")
    .agg(visits=("visit_id","count"),
         successful=("successful_visit","sum"),
         success_rate=("successful_visit","mean"),
         total_revenue=("transaction_amount","sum"))
    .round({"success_rate":3, "total_revenue":2})
)
print()

# ── Quartile distribution of visits ─────────────────────────────────────────
print("=" * 70)
print("VISIT QUARTILE DISTRIBUTION  (Territory × Truck Group × RFM Quartile)")
print("=" * 70)
print("""
RFM Quartile legend (within Territory × Truck Group peer set):
  Q4 = Top 25% of final_customer_score  (visit_freq_multiplier = 1.00)
  Q3 = 50th–75th percentile             (visit_freq_multiplier = 0.85)
  Q2 = 25th–50th percentile             (visit_freq_multiplier = 0.65)
  Q1 = Bottom 25%                        (visit_freq_multiplier = 0.45)
""")

quartile_summary = (
    visit_df.groupby(["cold_truck_required","rfm_quartile"])
    .agg(
        total_visits     = ("visit_id",           "count"),
        successful       = ("successful_visit",   "sum"),
        success_rate     = ("successful_visit",   "mean"),
        avg_txn_amount   = ("transaction_amount", lambda x: x[x>0].mean() if (x>0).any() else 0),
        total_revenue    = ("transaction_amount", "sum"),
    )
    .round({"success_rate":3,"avg_txn_amount":2,"total_revenue":2})
    .reset_index()
)
quartile_summary["cold_truck_required"] = quartile_summary["cold_truck_required"].map(
    {True:"Cold Truck", False:"Normal Truck"})
quartile_summary = quartile_summary.rename(columns={"cold_truck_required":"truck_group",
                                                     "rfm_quartile":"rfm_quartile (Q4=best)"})
display(quartile_summary)

print()
print("Quartile distribution by Territory:")
q_by_ter = (
    visit_df.merge(customer_df[["customer_id","territory_id"]], on="customer_id")
    .groupby(["territory_id","cold_truck_required","rfm_quartile"])
    .agg(visits=("visit_id","count"), revenue=("transaction_amount","sum"))
    .round(2).reset_index()
)
q_by_ter["cold_truck_required"] = q_by_ter["cold_truck_required"].map(
    {True:"Cold Truck", False:"Normal Truck"})
display(q_by_ter)

# ── Save ─────────────────────────────────────────────────────────────────────
from pathlib import Path
out = Path("saudi_master_data_output"); out.mkdir(parents=True, exist_ok=True)
visit_df.to_csv(out/"visit.csv", index=False)
print(f"\nSaved → {(out/'visit.csv').resolve()}")


visit_df shape  : (14245, 11)
Date range      : 2025-02-12  →  2026-05-10
Success rate    : 74.4%
Total revenue   : SAR 51,330,502

Cold-truck vs Normal-truck visit split:


,visits,successful,success_rate,total_revenue
cold_truck_required,,,,
False,11352,8394,0.739,37916397.54
True,2893,2202,0.761,13414104.84



VISIT QUARTILE DISTRIBUTION  (Territory × Truck Group × RFM Quartile)

RFM Quartile legend (within Territory × Truck Group peer set):
  Q4 = Top 25% of final_customer_score  (visit_freq_multiplier = 1.00)
  Q3 = 50th–75th percentile             (visit_freq_multiplier = 0.85)
  Q2 = 25th–50th percentile             (visit_freq_multiplier = 0.65)
  Q1 = Bottom 25%                        (visit_freq_multiplier = 0.45)



,truck_group,rfm_quartile (Q4=best),total_visits,successful,success_rate,avg_txn_amount,total_revenue
0,Normal Truck,1,1756,1059,0.603,1983.32,2100335.07
1,Normal Truck,2,2465,1690,0.686,3220.29,5442294.83
2,Normal Truck,3,3258,2519,0.773,3480.30,8766879.76
3,Normal Truck,4,3873,3126,0.807,6911.99,21606887.88
4,Cold Truck,1,462,236,0.511,1814.83,428299.13
5,Cold Truck,2,642,467,0.727,2635.87,1230950.09
6,Cold Truck,3,758,577,0.761,5615.00,3239855.72
7,Cold Truck,4,1031,922,0.894,9235.36,8514999.90



Quartile distribution by Territory:


,territory_id,cold_truck_required,rfm_quartile,visits,revenue
0,TER_DMM,Normal Truck,1,582,882242.03
1,TER_DMM,Normal Truck,2,809,1721343.11
2,TER_DMM,Normal Truck,3,1055,3008361.55
3,TER_DMM,Normal Truck,4,1223,7591481.64
4,TER_DMM,Cold Truck,1,172,197383.15
5,TER_DMM,Cold Truck,2,260,240736.63
6,TER_DMM,Cold Truck,3,272,1312860.10
7,TER_DMM,Cold Truck,4,388,1877934.50
8,TER_JED,Normal Truck,1,567,460621.65
9,TER_JED,Normal Truck,2,800,1919338.83



Saved → D:\Data Science\Basamh\JP_Yash\journey-planner\saudi_master_data_output\visit.csv


In [3]:

# ----- Visit table diagnostics -----
print("Visits per territory")
display(
    visit_df.merge(customer_df[["customer_id","territory_id"]], on="customer_id")
    .groupby("territory_id")
    .agg(total_visits=("visit_id","count"),
         successful=("successful_visit","sum"),
         success_rate=("successful_visit","mean"),
         total_revenue=("transaction_amount","sum"))
    .round({"success_rate":3,"total_revenue":2})
)

print("\nVisits per salesperson (top 10 by revenue)")
display(
    visit_df.groupby("sales_id")
    .agg(visits=("visit_id","count"),
         success_rate=("successful_visit","mean"),
         revenue=("transaction_amount","sum"))
    .sort_values("revenue",ascending=False).head(10)
    .round({"success_rate":3,"revenue":2})
)


Visits per territory


,total_visits,successful,success_rate,total_revenue
territory_id,,,,
TER_DMM,4761,3547,0.745,16832342.71
TER_JED,4754,3466,0.729,16874311.70
TER_RUH,4730,3583,0.758,17623847.97



Visits per salesperson (top 10 by revenue)


,visits,success_rate,revenue
sales_id,,,
SAL_JED_001,2705,0.771,12087671.55
SAL_DMM_001,2535,0.743,9820539.24
SAL_RUH_002,2198,0.750,8895345.28
SAL_RUH_003,1388,0.778,5331960.27
SAL_DMM_003,1053,0.742,3754734.50
SAL_RUH_001,1144,0.747,3396542.42
SAL_DMM_002,1173,0.753,3257068.97
SAL_JED_003,1115,0.703,2679869.27
SAL_JED_002,934,0.638,2106770.88



## Preview generated tables

Run the cells below after generation to inspect samples and distribution checks.


In [4]:

print("Territories")
display(territory_df)

print("Salespeople")
display(salesperson_df.head(10))

print("Vans (cold_truck_enabled flag)")
display(van_df.head(10))

print("Customers")
display(customer_df.head(10))

print("RFM scores (with cold_truck_group + rfm_segment_final)")
display(rfm_scores_df.head(10))


Territories


,territory_id,territory_name,warehouse_lat,warehouse_lng,warehouse_address,center_lat,center_lng,radius_km,default_salesperson,default_van
0,TER_RUH,Riyadh Central,24.5790,46.8237,"Industrial Area, Riyadh",24.7136,46.6753,25,SAL_RUH_001,VAN_RUH_001
1,TER_JED,Jeddah North,21.3429,39.2357,"Al Khomrah Logistics Area, Jeddah",21.5433,39.1728,22,SAL_JED_002,VAN_JED_002
2,TER_DMM,Dammam Metro,26.2926,50.1629,"2nd Industrial City, Dammam",26.4207,50.0888,20,SAL_DMM_002,VAN_DMM_002


Salespeople


,sales_id,name,territory_id,working_hours_per_day,shift_start_time,assigned_van,active_status,performance_multiplier
0,SAL_RUH_001,Arif Rahman,TER_RUH,8.0,09:00,VAN_RUH_001,True,1.04
1,SAL_RUH_002,Nasser Al-Dossari,TER_RUH,8.0,09:00,VAN_RUH_002,True,0.99
2,SAL_RUH_003,Abdullah Al-Qahtani,TER_RUH,8.0,09:00,VAN_RUH_003,True,0.97
3,SAL_JED_001,Ahmed Al-Anazi,TER_JED,8.0,09:00,VAN_JED_001,True,0.98
4,SAL_JED_002,Majed Al-Shammari,TER_JED,8.0,09:00,VAN_JED_002,True,1.14
5,SAL_JED_003,Imran Ahmed,TER_JED,8.0,09:00,VAN_JED_003,True,0.98
6,SAL_DMM_001,Khalid Al-Ghamdi,TER_DMM,8.0,09:00,VAN_DMM_001,True,0.97
7,SAL_DMM_002,Hassan Al-Yami,TER_DMM,8.0,09:00,VAN_DMM_002,True,1.13
8,SAL_DMM_003,Fahad Al-Otaibi,TER_DMM,8.0,09:00,VAN_DMM_003,True,0.88


Vans (cold_truck_enabled flag)


,van_id,registration_no,cold_truck_enabled,territory_id,active_status
0,VAN_RUH_001,RUB 4733,False,TER_RUH,True
1,VAN_RUH_002,RUC 4814,True,TER_RUH,True
2,VAN_RUH_003,RUM 5554,False,TER_RUH,True
3,VAN_RUH_004,RUL 3664,True,TER_RUH,True
4,VAN_JED_001,JEW 2169,True,TER_JED,True
5,VAN_JED_002,JEF 9751,False,TER_JED,True
6,VAN_JED_003,JEF 8573,False,TER_JED,True
7,VAN_JED_004,JEU 4598,True,TER_JED,True
8,VAN_DMM_001,DMZ 6168,True,TER_DMM,True
9,VAN_DMM_002,DMC 4456,False,TER_DMM,True


Customers


,customer_id,shop_name,gps_lat,gps_lng,locality,territory_id,customer_rating,review_rating,shop_category,cold_truck_required,volume_tier,payment_type,credit_limit,outstanding_balance,lifecycle_state,acquisition_date,preferred_visit_day,preferred_order_window
0,CUS_RUH_0001,Al Khaleej Mini Market,24.717094,46.672886,Olaya,TER_RUH,3,4.4,Mini Market,False,MED,credit,30590.81,3641.36,Active,2022-12-30,Thursday,Midday
1,CUS_RUH_0002,Al Othman Banquet Kitchen,24.684751,46.732198,Al Malaz,TER_RUH,3,3.9,Catering Kitchen,False,MED,cash,0.00,0.00,Active,2025-02-16,Tuesday,Midday
2,CUS_RUH_0003,Al Qassim Al Yasmin Grocery,24.844884,46.639722,Al Yasmin,TER_RUH,5,3.7,Grocery,False,LOW,cash,0.00,0.00,Active,2026-04-07,Saturday,Morning
3,CUS_RUH_0004,Al Malaz Cold Store,24.673710,46.729357,Al Malaz,TER_RUH,5,3.7,Cold Store,True,LOW,credit,4385.34,2877.09,Active,2021-12-16,Sunday,Morning
4,CUS_RUH_0005,Al Shehri Trading Restaurant,24.829825,46.606460,Al Yasmin,TER_RUH,5,3.1,Restaurant,False,MED,cash,0.00,0.00,Active,2025-10-30,Tuesday,Midday
5,CUS_RUH_0006,Al Nada Hotel Kitchen,24.820768,46.617633,Al Yasmin,TER_RUH,4,4.2,Hotel Kitchen,False,LOW,cash,0.00,0.00,New,2024-06-22,Saturday,Afternoon
6,CUS_RUH_0007,Al Sulaymaniyah Catering Kitchen,24.690975,46.693694,Al Sulaymaniyah,TER_RUH,5,4.8,Catering Kitchen,False,MED,cash,0.00,0.00,Active,2022-06-23,Saturday,Afternoon
7,CUS_RUH_0008,Al Safa Fresh Grill,24.708957,46.675437,Olaya,TER_RUH,4,3.2,Restaurant,False,MED,credit,42955.86,9489.06,New,2025-06-13,Monday,Morning
8,CUS_RUH_0009,Al Sulaymaniyah Cafe,24.695070,46.712575,Al Sulaymaniyah,TER_RUH,5,3.4,Cafe,False,MED,cash,0.00,0.00,Active,2021-11-05,Wednesday,Morning
9,CUS_RUH_0010,Al Shammari Trading Sweets & Bakery,24.767831,46.594751,Hittin,TER_RUH,5,3.8,Bakery,False,LOW,cash,0.00,0.00,Active,2025-05-18,Monday,Afternoon


RFM scores (with cold_truck_group + rfm_segment_final)


,customer_id,territory_id,cold_truck_group,last_successful_visit,recency,frequency,monetary,r_score,f_score,m_score,rfm_combined,seasonality_score,territory_score,locality_score,rating_score,final_customer_score,rfm_segment_final,customer_rank
0,CUS_DMM_0096,TER_DMM,Cold Truck,2026-05-10,28.0,6.0,61416.68,1.000000,0.857143,0.607976,0.821706,1.000000,0.251741,1.000000,0.50,0.803857,High,1
1,CUS_DMM_0036,TER_DMM,Cold Truck,2026-05-04,34.0,6.0,73267.45,0.903226,0.857143,0.725290,0.828553,0.992320,0.251741,0.202459,1.00,0.774537,High,2
2,CUS_DMM_0026,TER_DMM,Cold Truck,2026-05-10,28.0,7.0,22989.50,1.000000,1.000000,0.227578,0.742526,0.987455,0.251741,0.770527,0.25,0.720474,High,3
3,CUS_DMM_0014,TER_DMM,Cold Truck,2026-05-10,28.0,4.0,3564.68,1.000000,0.571429,0.035287,0.535572,0.982217,0.251741,1.000000,0.75,0.709068,High,4
4,CUS_DMM_0086,TER_DMM,Cold Truck,2026-05-05,33.0,5.0,3276.82,0.919355,0.714286,0.032438,0.555359,0.972651,0.251741,0.613349,1.00,0.700448,High,5
5,CUS_DMM_0067,TER_DMM,Cold Truck,2026-05-06,32.0,5.0,17957.34,0.935484,0.714286,0.177763,0.609178,0.746530,0.251741,1.000000,0.75,0.667804,High,6
6,CUS_DMM_0066,TER_DMM,Cold Truck,2026-04-23,45.0,3.0,36154.83,0.725806,0.428571,0.357904,0.504094,0.960215,0.251741,0.613349,0.75,0.651211,High,7
7,CUS_DMM_0054,TER_DMM,Cold Truck,2026-05-05,33.0,5.0,17423.88,0.919355,0.714286,0.172483,0.602041,0.656682,0.251741,0.613349,1.00,0.624330,Medium,8
8,CUS_DMM_0062,TER_DMM,Cold Truck,2026-04-20,48.0,2.0,37717.48,0.677419,0.285714,0.373373,0.445502,0.934677,0.251741,1.000000,0.25,0.608778,Medium,9
9,CUS_DMM_0011,TER_DMM,Cold Truck,2026-05-10,28.0,5.0,3081.97,1.000000,0.714286,0.030509,0.581598,0.745235,0.251741,0.770527,0.50,0.608436,Medium,10


In [5]:

print("Tier counts per territory")
display(customer_df.groupby(["territory_id","volume_tier"]).size().unstack(fill_value=0))

print("\nLifecycle distribution")
display(customer_df["lifecycle_state"].value_counts(normalize=True).rename("share").round(3))

print("\nRFM segment counts  — overall")
display(rfm_scores_df["rfm_segment_final"].value_counts())

print("\nRFM segment distribution by Territory × Truck Group")
print("(Top 30% = High, Middle 40% = Medium, Bottom 30% = Low within each group)")
seg_pivot = (
    rfm_scores_df.groupby(["territory_id","cold_truck_group","rfm_segment_final"])
    .size().unstack(fill_value=0)
    [["High","Medium","Low"]]
)
display(seg_pivot)

print("\nOutput files")
for path in sorted(__import__("pathlib").Path("saudi_master_data_output").glob("*")):
    print(path)


Tier counts per territory


volume_tier,HIGH,LOW,MED
territory_id,,,
TER_DMM,20,50,30
TER_JED,20,50,30
TER_RUH,20,50,30



Lifecycle distribution


lifecycle_state
Active     0.663
Dormant    0.120
At Risk    0.117
New        0.080
Churned    0.020
Name: share, dtype: float64


RFM segment counts  — overall


rfm_segment_final
Medium    120
High       93
Low        87
Name: count, dtype: int64


RFM segment distribution by Territory × Truck Group
(Top 30% = High, Middle 40% = Medium, Bottom 30% = Low within each group)


rfm_segment_final              High  Medium  Low
territory_id cold_truck_group                   
TER_DMM      Cold Truck           7      10    6
             Normal Truck        24      30   23
TER_JED      Cold Truck           7       9    6
             Normal Truck        24      31   23
TER_RUH      Cold Truck           5       7    4
             Normal Truck        26      33   25


Output files
saudi_master_data_output\config.csv
saudi_master_data_output\customer.csv
saudi_master_data_output\data_quality_report.json
saudi_master_data_output\distances_july01.xlsx
saudi_master_data_output\distances_jun15.xlsx
saudi_master_data_output\holiday.csv
saudi_master_data_output\holiday_1.csv
saudi_master_data_output\monthly_plan_new.xlsx
saudi_master_data_output\monthly_plan_new_1.xlsx
saudi_master_data_output\rfm_scores.csv
saudi_master_data_output\salesperson.csv
saudi_master_data_output\ter_customer.xlsx
saudi_master_data_output\territory.csv
saudi_master_data_output\unvisited.csv
saudi_master_data_output\van.csv
saudi_master_data_output\visit.csv



## Notes

- This notebook intentionally excludes route, visit, and basket-level transaction tables.
- The holiday dates are synthetic placeholders suitable for master-data demos. For production Saudi holiday calendars, replace those placeholder rows with an official calendar source.
- `generate_all(seed=42)` is deterministic for repeatable demos. Change the seed to create a different synthetic dataset.


In [6]:

# --- CASE 2: Schedule ONE specific territory only ---
from scheduler_2 import MultiSalespersonScheduler
scheduler = MultiSalespersonScheduler(solver_time_seconds=90)

result_ruh = scheduler.create_schedules(
    customer_df      = customer_df,
    rfm_scores_df    = rfm_scores_df,
    salesperson_df   = salesperson_df,
    holiday_df       = holiday_df,
    territory_df     = territory_df,
    config_df        = config_df,
    van_df           = van_df,
    month_start_date = "2026-07-01",
    territory_id     = "TER_RUH",      # only Riyadh
)


Territory TER_RUH: 100 customers, 3 active SPs
  Cold-truck customers : 16
  Normal-truck customers: 84

  [TER_RUH/cold] 16 customers × 1 SPs × 26 days → ~416 vars | timeout=300s
  [TER_RUH/cold] pairwise matrix: 17 nodes, avg_wh_leg=46min, avg_inter=7min, budget=480min
  [TER_RUH/cold] sp_daily_cap=16 stops/day (22min visit + 7min inter = 29min/stop, budget=480min)
  ✅ [TER_RUH/cold] OPTIMAL | obj=138832020053 | wall=0.2s

  [TER_RUH/normal] 84 customers × 2 SPs × 26 days → ~4368 vars | timeout=600s
  [TER_RUH/normal] pairwise matrix: 85 nodes, avg_wh_leg=45min, avg_inter=6min, budget=480min
  [TER_RUH/normal] sp_daily_cap=17 stops/day (22min visit + 6min inter = 28min/stop, budget=480min)
  ✅ [TER_RUH/normal] FEASIBLE | obj=717045216415 | wall=600.5s
  ⚠️  [TER_RUH/normal] 31 customer(s) NOT scheduled (capacity overflow — see unvisited_customers / under_visited_customers): ['CUS_RUH_0006', 'CUS_RUH_0015', 'CUS_RUH_0016', 'CUS_RUH_0022', 'CUS_RUH_0023'] …+26 more
remaining time for 

In [7]:
result = result_ruh

In [8]:
# All territories
from validate_scheduler import *
# checks = validate_schedule(result, customer_df, salesperson_df, van_df,
#                            territory_df, holiday_df, config_df, rfm_scores_df,
#                            month_start="2026-07-01", territory_id=None)

# One territory
checks = validate_schedule(result, customer_df, salesperson_df, van_df,
                           territory_df, holiday_df, config_df, rfm_scores_df,
                           month_start="2026-07-01", territory_id= "TER_RUH")

print (checks)

# Check if fully clean
is_valid = all(len(v) == 0 for v in checks.values())

# Get unvisited list directly
result.unvisited_customers.to_csv("unvisited.csv", index=False)


SCHEDULE VALIDATION REPORT  (scheduler v6 — adjust_priority)
Month: 2026-07-01  |  Scope: territory=TER_RUH
Rows: 1097  |  Customers: 71
Cold visits: 257  |  Normal visits: 840  |  Unvisited: 29  |  Under-visited: 26
  ✅ Cold truck requirement

  ℹ️  Min 1 visit per active customer (INFO) — 24 item(s):
     [INFO] Active customer CUS_RUH_0075 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0083 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0006 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0054 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0057 has 0 visits (capacity overflow — not a constraint breach)
     … and 19 more
  ✅ Max visits/month by segment
  ✅ Single salesperson per customer
  ✅ Daily customer count limit
  ✅ Daily total time limit (circuit-based)
  ✅ Holiday blocking
  ✅ Visit min

In [9]:
# --- CASE 7: Territory-wide map for one day (both truck groups) ---
from scheduler_2 import build_territory_day_map, build_stop_to_stop_map

m_all = build_territory_day_map(
    result        = result,
    territory_id  = "TER_RUH",
    schedule_date = "2026-07-15",
    truck_group   = None,      # None = show cold + normal together
)
display(m_all)

# Cold-only map
# m_cold = build_territory_day_map(
#     result        = result,
#     territory_id  = "TER_RUH",
#     schedule_date = "2026-07-15",
#     truck_group   = "normal",
# )
# display(m_cold)

m = build_stop_to_stop_map(result, "TER_RUH", "2026-07-30", "SAL_RUH_001")
m

In [10]:
# # def export_stop_to_stop_excel(
# #     result,
# #     territory_id: str,
# #     schedule_date,          # <<--- a date, not a filepath
# #     filepath: str = "stop_to_stop_distances.xlsx"
# # ):
    
# from scheduler import export_stop_to_stop_monthly_excel
# export_stop_to_stop_monthly_excel(
#     result,
#     territory_id="TER_RUH",
#     year_month="2025-07",
#     filepath="distances_july01.xlsx"
# )

In [11]:
import pandas as pd

# Define the output file name
excel_file_path = "monthly_plan_fixed_holiday.xlsx"

with pd.ExcelWriter(excel_file_path, engine="openpyxl") as writer:
    # 1. Save the main aggregated dataframes to their own sheets
    result.detailed_schedule.to_excel(
        writer, sheet_name="Detailed Schedule", index=False
    )
    result.daily_schedule.to_excel(writer, sheet_name="Daily Schedule", index=False)

    # 2. Loop through the salesperson dictionary and write each to a sheet
    for name, sp_result in result.salesperson_results.items():
        # Excel sheet names cannot exceed 31 characters
        safe_sheet_name = f"SP_{name}"

        # Check if the salesperson result object itself has dataframes to unpack
        # If it is a dictionary, use: sp_result_dict = sp_result
        # If it is another dataclass, convert it to a dict to read its attributes
        if hasattr(sp_result, "__dataclass_fields__"):
            from dataclasses import asdict

            sp_data = asdict(sp_result)
        elif isinstance(sp_result, dict):
            sp_data = sp_result
        else:
            # Fallback: tries to extract internal attributes if it's a custom class
            sp_data = vars(sp_result)

        # Look for any pandas DataFrames inside the salesperson's data to save
        for key, value in sp_data.items():
            if isinstance(value, pd.DataFrame):
                # Creates a unique sheet name like "SP_John_detailed"
                sub_sheet_name = f"SP_{name}_{key}"
                value.to_excel(writer, sheet_name=sub_sheet_name, index=False)

print(f"Successfully saved results to {excel_file_path}")


d:\Data Science\Basamh\JP_Yash\journey-planner\.venv\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Successfully saved results to monthly_plan_fixed_holiday.xlsx


In [12]:
result.unvisited_customers

,customer_id,shop_name,gps_lat,gps_lng,locality,territory_id,customer_rating,review_rating,shop_category,cold_truck_required,...,recency,frequency,monetary,r_score,f_score,m_score,seasonality_score,territory_score,locality_score,rating_score
0,CUS_RUH_0006,Al Nada Hotel Kitchen,24.820768,46.617633,Al Yasmin,TER_RUH,4,4.2,Hotel Kitchen,False,...,43.0,3.0,1842.46,0.722222,0.285714,0.018142,0.464968,1.0,1.000000,0.75
1,CUS_RUH_0015,Al Baraka Restaurant,24.666298,46.731369,Al Malaz,TER_RUH,5,3.2,Restaurant,False,...,40.0,2.0,4517.41,0.777778,0.142857,0.045521,0.259982,1.0,0.226109,1.00
2,CUS_RUH_0016,Al Ghamdi Trading Corner Market,24.659494,46.729392,Al Malaz,TER_RUH,1,4.6,Mini Market,False,...,35.0,4.0,2731.60,0.870370,0.428571,0.027243,0.487179,1.0,0.226109,0.00
3,CUS_RUH_0022,Al Safa Restaurant,24.687729,46.721272,Al Sulaymaniyah,TER_RUH,3,4.7,Restaurant,False,...,36.0,2.0,1336.01,0.851852,0.142857,0.012958,0.781384,1.0,0.365381,0.50
4,CUS_RUH_0023,Al Tazaj Catering Kitchen,24.826289,46.642652,Al Yasmin,TER_RUH,3,4.2,Catering Kitchen,False,...,56.0,2.0,3255.46,0.481481,0.142857,0.032605,0.336408,1.0,1.000000,0.50
5,CUS_RUH_0024,Al Shehri Trading Hospitality Supplies,24.756335,46.606682,Hittin,TER_RUH,4,4.8,Hotel Kitchen,False,...,40.0,4.0,11692.32,0.777778,0.428571,0.118960,0.463621,1.0,0.000000,0.75
6,CUS_RUH_0025,Al Mutairi Grill,24.826697,46.649614,Al Yasmin,TER_RUH,3,2.6,Restaurant,False,...,45.0,3.0,1943.54,0.685185,0.285714,0.019177,0.253955,1.0,1.000000,0.50
7,CUS_RUH_0034,Al Khaleej Banquet Kitchen,24.760958,46.584647,Hittin,TER_RUH,3,3.8,Catering Kitchen,False,...,43.0,2.0,1396.51,0.722222,0.142857,0.013577,0.264321,1.0,0.000000,0.50
8,CUS_RUH_0036,Al Rajhi Trading Corner Market,24.705750,46.734580,Al Sulaymaniyah,TER_RUH,4,4.1,Mini Market,False,...,41.0,4.0,4294.81,0.759259,0.428571,0.043243,0.633239,1.0,0.365381,0.75
9,CUS_RUH_0038,Al Waha Kitchen,24.826801,46.630947,Al Yasmin,TER_RUH,4,5.0,Restaurant,False,...,47.0,3.0,2184.97,0.648148,0.285714,0.021648,0.325376,1.0,1.000000,0.75


In [13]:
m = build_stop_to_stop_map(result, "TER_RUH", "2026-07-25", "SAL_RUH_001")
m

In [14]:
from scheduler_2 import export_under_visited_excel

export_under_visited_excel(result, "undervisted.xlsx")

Saved → undervisted.xlsx  (26 under-visited | 71 total customers)


'undervisted.xlsx'

In [15]:
# result.unvisited_customers

In [16]:
import importlib
import validate_scheduler

# Force reload the modified validation script
importlib.reload(validate_scheduler)
from validate_scheduler import validate_schedule

# Run the validation
checks = validate_schedule(
    result, 
    customer_df, 
    salesperson_df, 
    van_df,
    territory_df, 
    holiday_df, 
    config_df, 
    rfm_scores_df,
    month_start="2026-07-01", 
    territory_id="TER_RUH"
)



SCHEDULE VALIDATION REPORT  (scheduler v6 — adjust_priority)
Month: 2026-07-01  |  Scope: territory=TER_RUH
Rows: 1097  |  Customers: 71
Cold visits: 257  |  Normal visits: 840  |  Unvisited: 29  |  Under-visited: 26
  ✅ Cold truck requirement

  ℹ️  Min 1 visit per active customer (INFO) — 24 item(s):
     [INFO] Active customer CUS_RUH_0075 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0083 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0006 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0054 has 0 visits (capacity overflow — not a constraint breach)
     [INFO] Active customer CUS_RUH_0057 has 0 visits (capacity overflow — not a constraint breach)
     … and 19 more
  ✅ Max visits/month by segment
  ✅ Single salesperson per customer
  ✅ Daily customer count limit
  ✅ Daily total time limit (circuit-based)
  ✅ Holiday blocking
  ✅ Visit min